In [1]:
!pip install gradio transformers feedparser wikipedia sentencepiece --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.2 MB/s eta 0:00:00


In [2]:
import re, html, urllib.parse
from datetime import datetime

import feedparser
import gradio as gr
import torch
import wikipedia
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Faster on CPU. To go back, use "facebook/bart-large-cnn".
MODEL = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).eval()

TOP_WORDS = {"", "top", "top news", "headlines", "top headlines", "news", "latest news"}
print("Model loaded")

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model loaded


In [3]:
def split_title_source(title: str):
    """Google titles look like 'Headline text - Source Name'."""
    if " - " in title:
        head, src = title.rsplit(" - ", 1)
        return head.strip(), src.strip()
    return title.strip(), ""

def chunk_text(text: str, max_chars: int = 900):
    chunks, current = [], ""
    for s in re.split(r"(?<=[.!?])\s+", text):
        if current and len(current) + len(s) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += s + " "
    if current.strip():
        chunks.append(current.strip())
    return chunks

def safe_summarize(chunk: str):
    """Summarize with lengths scaled to the input size."""
    inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=1024)
    n = inputs["input_ids"].shape[1]
    if n < 30:  # too short to summarize, keep as is
        return chunk
    max_len = max(20, min(100, int(n * 0.6)))
    min_len = max(10, min(40, max_len - 10))
    with torch.no_grad():
        ids = model.generate(
            **inputs, max_length=max_len, min_length=min_len,
            num_beams=4, do_sample=False, early_stopping=True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True)

In [4]:
def fetch_google_news_rss(query: str, country="IN", max_items=10):
    try:
        base = "https://news.google.com/rss"
        params = f"hl=en-{country}&gl={country}&ceid={country}:en"
        if query.strip().lower() in TOP_WORDS:
            url = f"{base}?{params}"
        else:
            url = f"{base}/search?q={urllib.parse.quote_plus(query)}&{params}"
        feed = feedparser.parse(url)
        items = []
        for e in feed.entries[:max_items]:
            head, src = split_title_source(e.get("title", ""))
            src = (e.get("source", {}) or {}).get("title", src)
            items.append({"title": head, "source": src, "link": e.get("link", "")})
        return items
    except Exception:
        return []

def wikipedia_fallback(query: str):
    for title in wikipedia.search(query)[:3]:
        try:
            page = wikipedia.page(title, auto_suggest=False)
            return page.content[:2500], page.url
        except Exception:  # disambiguation / missing page, try next
            continue
    return None, None

In [5]:
def ask_news(query, history=None):
    try:
        query = (query or "").strip()
        items = fetch_google_news_rss(query)
        timestamp = datetime.now().strftime("%d %b %Y, %H:%M")

        if items:
            text = ". ".join(it["title"].rstrip(".") for it in items) + "."
            chunks = chunk_text(text)
            summary = " ".join(safe_summarize(c) for c in chunks)

            lines = [
                f"{i}. [{it['title']}]({it['link']})"
                + (f" — *{it['source']}*" if it["source"] else "")
                for i, it in enumerate(items[:8], 1)
            ]
            return (
                f"📰 **Inshorts-style Summary:** {summary}\n\n"
                f"🔗 **Headlines:**\n" + "\n".join(lines) + "\n\n"
                f"💡 *Summarized from {len(items)} live Google News headlines "
                f"using {MODEL.split('/')[-1]}.*\n"
                f"🕒 {timestamp}"
            )

        text, url = wikipedia_fallback(query)
        if not text:
            return "😕 Sorry, no recent news found for that query."
        summary = " ".join(safe_summarize(c) for c in chunk_text(text)[:3])
        return (
            f"📚 *No recent news found, here's a background summary:*\n\n{summary}\n\n"
            f"🔗 {url}\n🕒 {timestamp}"
        )
    except Exception as e:
        return f"⚠️ Error: {e}"

# Quick test before launching the UI
print(ask_news("Top headlines")[:500])

📰 **Inshorts-style Summary:**  Nana Patekar: Bollywood actor dies at 75 in Goa home . Three killed, 36 injured in Houthi attacks on Saudi airports, Riyadh says . Delhi to deploy 23,000 CAPF personnel. Forces pulled from J&K, Naxal ops. Supreme Court Adjourns Punjab Plea Challenging CBI Probe Against CM Bhagwant Mann's OSD Till Tomorrow.

🔗 **Headlines:**
1. [Nana Patekar: Bollywood actor dies at 75 in Goa home](https://news.google.com/rss/articles/CBMiW0FVX3lxTFB4Tnp5MFI4c1lxZ195NVJOUHpsRjNwT3pX


In [8]:
iface = gr.ChatInterface(
    fn=ask_news,
    title="🧠 Inshorts-style AI News Assistant (No API Key)",
    description="Ask for the latest news on any topic. Type 'top headlines' for general news.",
    examples=[["Top headlines"], ["World news"], ["AI trends"],
              ["Space exploration"], ["Indian politics"]],
)
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://17af6957d6dca4c100.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
